# PyTorch 2.0 編譯優化：torch.compile()
:label:`sec_torch_compile`

PyTorch 2.0 引入了革命性的 `torch.compile()` 功能，這是 PyTorch 近年來最重要的性能突破。通過即時編譯（JIT compilation），它可以在不改變代碼的情況下實現：

- **訓練加速 30-200%**：取決於模型架構
- **推理加速 2-10x**：尤其是小批次推理
- **自動優化**：圖級別優化、kernel fusion、記憶體規劃
- **保持靈活性**：仍然支持動態控制流

本章將深入探討 torch.compile() 的工作原理、使用方法和優化技巧。

## 目錄
1. [torch.compile() 基礎](#1-torchcompile-基礎)
2. [編譯器後端詳解](#2-編譯器後端詳解)
3. [TorchDynamo 工作原理](#3-torchdynamo-工作原理)
4. [性能優化策略](#4-性能優化策略)
5. [常見問題與調試](#5-常見問題與調試)
6. [實戰案例分析](#6-實戰案例分析)
7. [與其他優化技術結合](#7-與其他優化技術結合)

In [ ]:
# 導入必要的庫
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import numpy as np
from torchvision import models
import warnings

# 檢查 PyTorch 版本
print(f"PyTorch 版本: {torch.__version__}")

if torch.__version__ < '2.0.0':
    warnings.warn("torch.compile() 需要 PyTorch 2.0 或更高版本")
    HAS_COMPILE = False
else:
    HAS_COMPILE = True
    print("✓ 支持 torch.compile()")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用設備: {device}")

## 1. torch.compile() 基礎

### 1.1 最簡單的使用方式

`torch.compile()` 的使用極其簡單 - 只需一行代碼包裝你的模型：

In [ ]:
# 定義一個簡單的模型
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, 10)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# 傳統方式
model = SimpleModel().to(device)

# 編譯優化 - 就這麼簡單！
if HAS_COMPILE:
    model_compiled = torch.compile(model)
    print("✓ 模型編譯成功")
else:
    model_compiled = model
    print("使用未編譯的模型（PyTorch < 2.0）")

### 1.2 快速性能比較

In [ ]:
def benchmark_model(model, input_tensor, num_iterations=100, warmup=10):
    """基準測試模型性能"""
    # 預熱
    for _ in range(warmup):
        _ = model(input_tensor)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    # 測試
    start = time.time()
    for _ in range(num_iterations):
        _ = model(input_tensor)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    elapsed = time.time() - start
    return elapsed / num_iterations

# 準備測試數據
batch_size = 32
test_input = torch.randn(batch_size, 3, 224, 224, device=device)

# 比較性能
print("\n" + "="*60)
print("性能比較: 普通模型 vs 編譯模型")
print("="*60)

model_normal = SimpleModel().to(device)
model_normal.eval()

with torch.no_grad():
    time_normal = benchmark_model(model_normal, test_input)
    print(f"普通模型: {time_normal*1000:.2f} ms/iteration")
    
    if HAS_COMPILE:
        model_compiled = torch.compile(model_normal)
        time_compiled = benchmark_model(model_compiled, test_input)
        print(f"編譯模型: {time_compiled*1000:.2f} ms/iteration")
        print(f"加速比: {time_normal/time_compiled:.2f}x")
    else:
        print("需要 PyTorch 2.0+ 來使用 torch.compile()")

## 2. 編譯器後端詳解

`torch.compile()` 支持多種後端，每種都有不同的優化策略和適用場景。

### 2.1 可用的後端

| 後端 | 說明 | 適用場景 | 速度 | 兼容性 |
|------|------|----------|------|--------|
| `inductor` | 默認後端，TorchInductor | 訓練和推理 | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| `aot_eager` | AOTAutograd + Eager | 調試 | ⭐⭐ | ⭐⭐⭐⭐⭐ |
| `cudagraphs` | CUDA Graphs | GPU 推理 | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ |
| `onnxrt` | ONNX Runtime | 跨平台部署 | ⭐⭐⭐ | ⭐⭐⭐ |
| `tensorrt` | NVIDIA TensorRT | GPU 推理 | ⭐⭐⭐⭐⭐ | ⭐⭐ |

In [ ]:
if HAS_COMPILE:
    # 測試不同的後端
    backends_to_test = [
        ('inductor', '默認後端，最佳平衡'),
        ('aot_eager', 'AOT + Eager，用於調試'),
    ]
    
    # 如果是 CUDA，添加 cudagraphs
    if torch.cuda.is_available():
        backends_to_test.append(('cudagraphs', 'CUDA Graphs，推理優化'))
    
    print("\n" + "="*70)
    print("不同後端性能比較")
    print("="*70)
    
    model_base = SimpleModel().to(device).eval()
    
    results = {}
    
    with torch.no_grad():
        for backend_name, description in backends_to_test:
            try:
                print(f"\n測試後端: {backend_name} ({description})")
                model_compiled = torch.compile(model_base, backend=backend_name)
                
                # 預熱並測試
                elapsed = benchmark_model(model_compiled, test_input, num_iterations=50)
                results[backend_name] = elapsed
                
                print(f"  時間: {elapsed*1000:.2f} ms/iteration")
            except Exception as e:
                print(f"  ✗ 後端不可用: {e}")
    
    # 找出最快的後端
    if results:
        best_backend = min(results, key=results.get)
        print(f"\n✓ 最快的後端: {best_backend} ({results[best_backend]*1000:.2f} ms)")
else:
    print("需要 PyTorch 2.0+ 來測試不同後端")

### 2.2 編譯模式

`torch.compile()` 提供三種優化級別：

In [ ]:
if HAS_COMPILE:
    print("\n編譯模式詳解：\n")
    
    modes = [
        ('default', '默認模式，平衡速度和編譯時間'),
        ('reduce-overhead', '減少開銷，適合小批次'),
        ('max-autotune', '最大化性能，編譯時間長'),
    ]
    
    for mode, desc in modes:
        print(f"• {mode:20s}: {desc}")
    
    # 示例：不同模式的使用
    print("\n使用示例：")
    print("""
# 默認模式（推薦）
model_default = torch.compile(model)

# 減少開銷模式（小批次推理）
model_low_overhead = torch.compile(model, mode='reduce-overhead')

# 最大性能模式（可接受長編譯時間）
model_max_perf = torch.compile(model, mode='max-autotune')
    """)
else:
    print("需要 PyTorch 2.0+ 來使用不同編譯模式")

## 3. TorchDynamo 工作原理

### 3.1 編譯流程

```
Python 代碼
    ↓
TorchDynamo (捕獲計算圖)
    ↓
AOTAutograd (Ahead-of-Time 自動微分)
    ↓
TorchInductor (代碼生成)
    ↓
優化的 C++/CUDA Kernel
```

### 3.2 圖捕獲與優化

In [ ]:
# 演示圖捕獲過程
if HAS_COMPILE:
    import torch._dynamo as dynamo
    
    # 定義一個包含控制流的模型
    class DynamicModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.linear1 = nn.Linear(10, 20)
            self.linear2 = nn.Linear(20, 10)
        
        def forward(self, x, threshold=0.5):
            x = self.linear1(x)
            x = F.relu(x)
            
            # 動態控制流
            if x.mean() > threshold:
                x = x * 2
            
            x = self.linear2(x)
            return x
    
    # 查看編譯過程
    print("=== 查看編譯過程 ===")
    
    model = DynamicModel()
    
    # 啟用詳細日誌
    # torch._dynamo.config.verbose = True
    
    model_compiled = torch.compile(model)
    
    # 第一次運行會觸發編譯
    x = torch.randn(5, 10)
    print("\n第一次運行（觸發編譯）...")
    output = model_compiled(x)
    
    # 第二次運行使用緩存的編譯結果
    print("第二次運行（使用緩存）...")
    output = model_compiled(x)
    
    print("✓ 編譯完成")
else:
    print("需要 PyTorch 2.0+ 來查看編譯過程")

## 4. 性能優化策略

### 4.1 Kernel Fusion（算子融合）

編譯器會自動合併多個操作，減少記憶體訪問：

In [ ]:
# 演示 Kernel Fusion 的好處
class UnfusedOps(nn.Module):
    """多個獨立操作"""
    def forward(self, x):
        # 這些操作可以融合
        x = x + 1.0
        x = x * 2.0
        x = torch.relu(x)
        x = x / 3.0
        return x

print("\n=== Kernel Fusion 效果演示 ===")

model_unfused = UnfusedOps()
x = torch.randn(1000, 1000, device=device)

# 未編譯版本
with torch.no_grad():
    time_unfused = benchmark_model(model_unfused, x, num_iterations=1000)
    print(f"未編譯: {time_unfused*1000:.3f} ms")

if HAS_COMPILE:
    # 編譯版本（自動融合）
    model_fused = torch.compile(model_unfused)
    
    with torch.no_grad():
        time_fused = benchmark_model(model_fused, x, num_iterations=1000)
        print(f"編譯後: {time_fused*1000:.3f} ms")
        print(f"加速比: {time_unfused/time_fused:.2f}x")
        print(f"\n說明: 編譯器將 4 個操作融合為 1 個 kernel")
else:
    print("需要 PyTorch 2.0+ 來演示 Kernel Fusion")

### 4.2 記憶體規劃優化

In [ ]:
# 演示記憶體優化
class MemoryIntensiveModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(512, 512) for _ in range(10)])
    
    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        return x

if torch.cuda.is_available() and HAS_COMPILE:
    print("\n=== 記憶體使用比較 ===")
    
    model = MemoryIntensiveModel().to(device)
    x = torch.randn(128, 512, device=device)
    
    # 測試未編譯版本
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        _ = model(x)
    mem_uncompiled = torch.cuda.max_memory_allocated() / 1024**2
    
    # 測試編譯版本
    model_compiled = torch.compile(model)
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        _ = model_compiled(x)
    mem_compiled = torch.cuda.max_memory_allocated() / 1024**2
    
    print(f"未編譯: {mem_uncompiled:.2f} MB")
    print(f"編譯後: {mem_compiled:.2f} MB")
    print(f"節省: {(1 - mem_compiled/mem_uncompiled)*100:.1f}%")
else:
    print("需要 CUDA GPU 和 PyTorch 2.0+ 來測試記憶體優化")

## 5. 常見問題與調試

### 5.1 圖中斷（Graph Break）

某些 Python 操作會導致圖中斷，影響性能：

In [ ]:
if HAS_COMPILE:
    import torch._dynamo as dynamo
    
    # 會導致圖中斷的操作
    class ProblematicModel(nn.Module):
        def forward(self, x):
            x = x * 2
            
            # 圖中斷點 1: Python 列表操作
            # x_list = x.tolist()  # 這會中斷圖
            
            # 圖中斷點 2: 打印
            # print(x.shape)  # 這會中斷圖
            
            x = x + 1
            return x
    
    print("\n=== 檢測圖中斷 ===")
    print("\n使用 explain() 來查看編譯信息：\n")
    
    model = ProblematicModel()
    
    # 解釋編譯過程
    explanation = torch._dynamo.explain(model)(torch.randn(10, 10))
    print(explanation)
    
    print("\n提示: 盡量避免以下操作以減少圖中斷:")
    print("  • 打印語句 (print)")
    print("  • Python 列表操作 (.tolist(), .item())")
    print("  • 複雜的控制流")
    print("  • 不支持的第三方庫")
else:
    print("需要 PyTorch 2.0+ 來檢測圖中斷")

### 5.2 調試編譯問題

In [ ]:
if HAS_COMPILE:
    print("\n=== 調試工具 ===")
    print("""
1. 查看編譯統計信息:
   torch._dynamo.reset()
   # 運行模型
   print(torch._dynamo.utils.compile_times())

2. 禁用特定優化進行調試:
   torch._dynamo.config.suppress_errors = True
   
3. 查看生成的代碼:
   torch._dynamo.config.verbose = True
   
4. 如果遇到錯誤，可以逐步回退:
   # 完全禁用編譯
   torch._dynamo.config.disable = True
   
   # 或使用更保守的模式
   model = torch.compile(model, backend='aot_eager')
    """)
    
    # 實用的調試函數
    def debug_compile(model, input_data):
        """調試編譯問題"""
        print("\n開始調試編譯...")
        
        # 重置狀態
        torch._dynamo.reset()
        
        try:
            # 嘗試編譯
            compiled_model = torch.compile(model)
            _ = compiled_model(input_data)
            print("✓ 編譯成功")
            return compiled_model
        except Exception as e:
            print(f"✗ 編譯失敗: {e}")
            print("\n嘗試使用 aot_eager 後端...")
            try:
                compiled_model = torch.compile(model, backend='aot_eager')
                _ = compiled_model(input_data)
                print("✓ aot_eager 編譯成功")
                return compiled_model
            except Exception as e2:
                print(f"✗ aot_eager 也失敗: {e2}")
                return model
    
    print("\n調試函數 debug_compile() 已定義")
else:
    print("需要 PyTorch 2.0+ 來使用調試工具")

## 6. 實戰案例分析

### 6.1 ResNet-50 完整優化

In [ ]:
if HAS_COMPILE:
    print("\n" + "="*70)
    print("ResNet-50 完整優化流程")
    print("="*70)
    
    # 加載模型
    model = models.resnet50(pretrained=False).to(device)
    model.eval()
    
    # 測試數據
    batch_sizes = [1, 8, 32]
    input_size = (3, 224, 224)
    
    results = {}
    
    for batch_size in batch_sizes:
        print(f"\n測試批次大小: {batch_size}")
        print("-" * 50)
        
        x = torch.randn(batch_size, *input_size, device=device)
        
        # 基線性能
        with torch.no_grad():
            time_baseline = benchmark_model(model, x, num_iterations=100)
        
        # 不同編譯模式
        modes = ['default', 'reduce-overhead', 'max-autotune']
        
        batch_results = {'baseline': time_baseline}
        
        for mode in modes:
            try:
                torch._dynamo.reset()
                model_compiled = torch.compile(model, mode=mode)
                
                with torch.no_grad():
                    time_compiled = benchmark_model(model_compiled, x, num_iterations=100)
                
                batch_results[mode] = time_compiled
                speedup = time_baseline / time_compiled
                
                print(f"  {mode:20s}: {time_compiled*1000:6.2f} ms ({speedup:.2f}x)")
            except Exception as e:
                print(f"  {mode:20s}: 失敗 - {e}")
        
        results[batch_size] = batch_results
    
    # 總結
    print("\n" + "="*70)
    print("優化建議總結")
    print("="*70)
    print("""
• 小批次 (batch_size=1): 使用 'reduce-overhead' 模式
• 中等批次 (batch_size=8-32): 使用 'default' 模式
• 大批次 (batch_size>32): 考慮 'max-autotune' 模式
• 生產環境: 預先編譯並保存模型
    """)
else:
    print("需要 PyTorch 2.0+ 來運行 ResNet-50 優化案例")

### 6.2 Transformer 模型優化

In [ ]:
# 簡單的 Transformer 層
class SimpleTransformer(nn.Module):
    def __init__(self, d_model=512, nhead=8):
        super().__init__()
        self.attention = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model)
        )
    
    def forward(self, x):
        # Self-attention
        attn_out, _ = self.attention(x, x, x)
        x = self.norm1(x + attn_out)
        
        # Feed-forward
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)
        
        return x

if HAS_COMPILE:
    print("\n=== Transformer 優化 ===")
    
    model = SimpleTransformer().to(device)
    model.eval()
    
    # 不同序列長度
    seq_lengths = [64, 128, 256]
    batch_size = 16
    
    print(f"\n批次大小: {batch_size}")
    print("-" * 50)
    
    for seq_len in seq_lengths:
        x = torch.randn(batch_size, seq_len, 512, device=device)
        
        with torch.no_grad():
            # 基線
            time_base = benchmark_model(model, x, num_iterations=50)
            
            # 編譯
            torch._dynamo.reset()
            model_compiled = torch.compile(model)
            time_compiled = benchmark_model(model_compiled, x, num_iterations=50)
            
            speedup = time_base / time_compiled
            print(f"序列長度 {seq_len:3d}: {time_base*1000:6.2f} ms -> "
                  f"{time_compiled*1000:6.2f} ms ({speedup:.2f}x)")
else:
    print("需要 PyTorch 2.0+ 來優化 Transformer")

## 7. 與其他優化技術結合

### 7.1 torch.compile() + AMP

In [ ]:
if HAS_COMPILE and torch.cuda.is_available():
    from torch.cuda.amp import autocast
    
    print("\n=== torch.compile() + 混合精度訓練 ===")
    
    model = SimpleModel().to(device)
    x = torch.randn(32, 3, 224, 224, device=device)
    
    # 1. 僅 compile
    model_compile = torch.compile(model)
    with torch.no_grad():
        time_compile = benchmark_model(model_compile, x)
    
    # 2. 僅 AMP
    with torch.no_grad(), autocast():
        time_amp = benchmark_model(model, x)
    
    # 3. Compile + AMP
    with torch.no_grad(), autocast():
        time_both = benchmark_model(model_compile, x)
    
    print(f"\n僅 Compile:     {time_compile*1000:.2f} ms")
    print(f"僅 AMP:         {time_amp*1000:.2f} ms")
    print(f"Compile + AMP:  {time_both*1000:.2f} ms")
    print(f"\n組合加速比: {time_compile / time_both:.2f}x (相對於僅 Compile)")
    
    print("\n✓ 推薦: 同時使用 torch.compile() 和 AMP 獲得最佳性能")
else:
    print("需要 CUDA GPU 和 PyTorch 2.0+ 來測試組合優化")

### 7.2 最佳實踐總結

In [ ]:
print("""
=================================================================
torch.compile() 最佳實踐
=================================================================

✓ 推薦做法:

1. 訓練時:
   model = torch.compile(model)
   optimizer = torch.optim.Adam(model.parameters())
   scaler = GradScaler()  # 配合 AMP

2. 推理時:
   model.eval()
   model = torch.compile(model, mode='reduce-overhead')
   with torch.no_grad(), autocast():
       output = model(input)

3. 部署時:
   # 選擇合適的後端
   if use_tensorrt:
       model = torch.compile(model, backend='tensorrt')
   elif use_onnx:
       model = torch.compile(model, backend='onnxrt')

✗ 避免:

1. 在編譯的模型中使用 print() 或 .item()
2. 頻繁改變輸入形狀（會重新編譯）
3. 在小模型上過度優化（編譯開銷可能超過收益）
4. 忘記預熱（第一次運行會包含編譯時間）

⚡ 性能提示:

• GPU 訓練: 預期 30-100% 加速
• GPU 推理: 預期 2-5x 加速
• CPU: 加速效果有限，主要受益於 operator fusion
• Transformer: 通常有 50-150% 加速
• CNN: 通常有 20-80% 加速

=================================================================
""")

## 總結

### 關鍵要點

1. **torch.compile() 是 PyTorch 2.0 最重要的特性**，一行代碼即可獲得顯著加速
2. **選擇合適的後端和模式**：
   - `inductor`: 默認，平衡性能
   - `cudagraphs`: GPU 推理最快
   - `reduce-overhead`: 小批次優化
3. **避免圖中斷**：減少 Python 控制流和副作用操作
4. **與其他優化結合**：AMP + compile 可獲得更大提升
5. **注意首次編譯開銷**：生產環境應預先編譯

### 下一步學習

- 深入學習 TorchDynamo 和 TorchInductor 原理
- 探索自定義後端開發
- 結合量化技術進一步優化
- 學習使用 AOTInductor 進行靜態編譯

### 參考資源

- [PyTorch 2.0 官方文檔](https://pytorch.org/get-started/pytorch-2.0/)
- [TorchDynamo 深入解析](https://pytorch.org/docs/stable/dynamo/)
- [性能調優指南](https://pytorch.org/tutorials/recipes/recipes/tuning_guide.html)

## 練習

1. 在你的項目中嘗試 `torch.compile()`，記錄性能提升
2. 比較不同編譯模式在你的模型上的效果
3. 使用 `torch._dynamo.explain()` 分析圖中斷原因
4. 結合 AMP 和 compile，找到最佳配置
5. 嘗試自定義後端（進階）